# ACE RAG Agent — Production

Registers `RagAgentModel` to Unity Catalog and deploys `ace-rag-agent-endpoint`.

The RAG agent answers policy questions by searching `policy_documents_index` via Vector Search and calling Claude Sonnet to produce a grounded, cited answer.
`LLM_ENDPOINT` is set to Sonnet.

| Cell | Purpose |
|------|---------|
| 1 | Install `databricks-vectorsearch` and `mlflow` |
| 2 | Central config — `LLM_ENDPOINT` = Sonnet |
| 3 | Register `RagAgentModel` pyfunc to Unity Catalog |
| 4 | Deploy `ace-rag-agent-endpoint` |

> **Prod note:** Dependency check, notebook test function, unit tests, and model verification cells removed.
> Wait for Cell 4 to print **"ace-rag-agent-endpoint is READY"** before running `page_context_agent.ipynb`.


In [ ]:
# ================================================================
# CELL 1 -- Install Dependencies
# ================================================================
#
# PURPOSE:
#   Installs the two packages needed by this notebook before any
#   other cell can run.
#
# PACKAGES:
#   databricks-vectorsearch
#     Provides VectorSearchClient used in Cell 3 (dependency check),
#     Cell 4 (interactive rag_agent function), and inside
#     RagAgentModel.predict() at serving time. Not included in the
#     default Databricks ML runtime -- must be installed explicitly.
#
#   mlflow
#     Used for:
#       - Experiment tracking: logs each rag_agent() call as an MLflow
#         run (retrieved context, answer, metrics) in Cell 4
#       - Model registration: Cell 6 logs RagAgentModel as a pyfunc
#         model and registers it in Unity Catalog
#       - Model loading: Cell 7 loads the registered model for verification
#       - Deployment: Cell 8 queries the MLflow model registry to get
#         the version number before deploying the serving endpoint
#     A recent mlflow version is required for Unity Catalog registry
#     support (mlflow.set_registry_uri("databricks-uc")).
#
# KERNEL RESTART:
#   `dbutils.library.restartPython()` restarts the Python interpreter
#   so the newly installed packages are importable. After restart,
#   begin from Cell 2 -- do NOT re-run Cell 1.
# ================================================================

%pip install databricks-vectorsearch mlflow
dbutils.library.restartPython()

In [ ]:
# ================================================================
# CELL 2 -- Central Configuration
# ================================================================
#
# PURPOSE:
#   All settings for the RAG agent live here. Cells 3-8 read from
#   these variables -- nothing is hardcoded elsewhere. To tune the
#   agent, edit only this cell and re-run from Cell 3.
#
# VARIABLE REFERENCE:
#
# CATALOG / SCHEMA
#   Unity Catalog namespace. Must match 01_infrastructure exactly.
#
# VS_ENDPOINT_NAME
#   The Databricks Vector Search endpoint created in 01_infrastructure.
#   Hosts the policy_documents_index.
#
# VS_INDEX_NAME
#   The fully-qualified VS index. This is what the RAG agent searches.
#   Built from policy_documents by 02_pdf_ingestion.
#
# TABLE_POLICY
#   Used in Cell 3 (dependency check) to verify the knowledge base
#   has been populated. Not queried at inference time -- the agent
#   uses the VS index, not the raw table.
#
# LLM_ENDPOINT
#   Databricks external model endpoint for Claude Haiku 4.5.
#   All LLM calls go through Databricks gateway -- data never leaves
#   the Azure tenant. Do not change without compliance sign-off.
#
# NUM_RESULTS (default: 4)
#   Number of policy chunks fetched per similarity search.
#   Raise to 9-10 for broad questions spanning multiple sections.
#   Lower to 4-5 for narrow factual queries.
#   Too many (12+) causes context window bloat.
#
# MAX_TOKENS (default: 600)
#   Maximum output tokens from Claude Haiku.
#   Raise to 2048 if answers are being cut off mid-response.
#
# EXPERIMENT_NAME
#   MLflow experiment where Cell 4 logs interactive test runs.
#   Uses the current user directory -- always exists, no setup needed.
#
# REGISTERED_MODEL
#   Unity Catalog path where RagAgentModel is registered.
#   Format: <catalog>.<schema>.<model_name>
#
# MODEL_ALIAS
#   champion -- the alias Cell 6 sets on the newly registered version.
#   The Supervisor references the model by this alias so it always
#   points to the latest promoted version without hardcoding a number.
#
# SYSTEM_PROMPT
#   The most important tuning lever for answer quality.
#   Key rules enforced:
#     - Answer ONLY from retrieved context (no general LLM knowledge)
#     - Cite every factual claim with [Source N]
#     - Reproduce exact figures (LTV, DTI, dollar amounts, dates)
#     - Flag contradictions between sources explicitly
#     - Use exact policy language (must vs should)
#   To improve quality: edit this, re-run Cell 5, compare before
#   re-registering.
# ================================================================

CATALOG              = "ace_theorem"
SCHEMA               = "chat"

VS_ENDPOINT_NAME     = "ace-chat-vs-endpoint"
VS_INDEX_NAME        = f"{CATALOG}.{SCHEMA}.policy_documents_index"
TABLE_POLICY         = f"{CATALOG}.{SCHEMA}.policy_documents"

# Claude Haiku 4.5 via Databricks gateway -- data stays in Azure
LLM_ENDPOINT         = "databricks-claude-sonnet-4-5"

NUM_RESULTS          = 4      # Chunks returned per similarity search (reduced from 7 for latency)
MAX_TOKENS           = 600    # Max LLM output tokens per response (reduced from 1024 for latency)

EXPERIMENT_NAME      = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/ace-rag-agent"
REGISTERED_MODEL     = f"{CATALOG}.{SCHEMA}.rag_agent"
MODEL_ALIAS          = "champion"

# System prompt -- edit and re-run Cell 5 to evaluate before registering
SYSTEM_PROMPT = """You are a lending policy expert for Vantage Bank. Your role is to answer questions using only the provided policy excerpts and relevant prior conversation context.

## Rules
1. Use only the provided excerpts — never use general banking knowledge or outside information.
2. If the question is unrelated to bank lending policy, decline to answer.
3. Cite every factual statement with [Source N].
4. Copy figures (LTV, DTI, thresholds, dates, limits, approval levels) exactly — no rounding, rewriting, or paraphrasing.
5. If the answer is not covered by the excerpts, state exactly: "The available policy documents do not cover this. Please refer to the appropriate policy owner or the Compliance team."
6. If sources contradict, state exactly: "Note: [Source N] and [Source M] contain conflicting guidance on this point."
7. Use the document's exact wording, including policy-strength terms such as must, should, may, and requires.
8. Never speculate, infer unstated requirements, or soften policy language.
9. Do not cite a source unless it directly supports the statement it follows.
10. If the answer depends on missing context not provided in the excerpts, state that the available excerpts are insufficient to determine the answer definitively.
11. Use prior conversation only to clarify references — never as policy evidence and never in place of the provided excerpts.
12. If multiple consecutive statements come from the same document and section, cite [Source N] once at the end of the last statement in that group — do not repeat the same citation after every line.

## Response format

**Answer**
- Give a direct answer in 1–2 sentences.
- If the answer is partial, conditional, or limited by missing information, say that clearly.

**Policy Basis**
- Present supporting policy points as bullets.
- Use one bullet per requirement, exception, threshold, approval rule, prohibition, or condition. [Source N]

**Key Figures** *(omit if none)*
- Metric: exact value [Source N]
- Copy numeric values exactly as written.

**Limitations / Missing Context** *(omit if none)*

**Conflicting Guidance** *(omit if none)*
- Note: [Source N] and [Source M] contain conflicting guidance on this point.

**Sources**
- [Source N]: Document name — Section name
"""

print("=" * 55)
print("ACE RAG Agent -- Config")
print("=" * 55)
print(f"  VS endpoint      : {VS_ENDPOINT_NAME}")
print(f"  VS index         : {VS_INDEX_NAME}")
print(f"  LLM endpoint     : {LLM_ENDPOINT}")
print(f"  Num results      : {NUM_RESULTS}")
print(f"  Max tokens       : {MAX_TOKENS}")
print(f"  Experiment       : {EXPERIMENT_NAME}")
print(f"  Registered model : {REGISTERED_MODEL}")
print(f"  Model alias      : {MODEL_ALIAS}")
print("=" * 55)


In [ ]:
# ================================================================
# CELL 3 -- Register RagAgentModel to Unity Catalog (Production)
# ================================================================
#
# PURPOSE:
#   Defines the production RagAgentModel class, logs it as an MLflow
#   pyfunc model, and registers it to Unity Catalog. This is the code
#   that actually runs inside the Databricks Model Serving container.
#
# WHY A SEPARATE CLASS FROM CELL 4:
#   The RagAgentModel class must be 100% self-contained -- all imports,
#   constants, and logic live inside the class. This is required because
#   the serving container is an isolated Python environment with no
#   access to notebook-level variables (CATALOG, SYSTEM_PROMPT, etc.).
#   The notebook-level rag_agent() in Cell 4 is for interactive testing
#   only; this class is the production artifact.
#
# AUTHENTICATION (load_context + _get_token):
#   load_context() runs once when the container starts up. It reads
#   DATABRICKS_HOST from env vars and initialises the MLflow deployments
#   client for LLM calls.
#
#   _get_token() is called on every predict() invocation. It tries SP
#   credentials first (SP_CLIENT_ID, SP_CLIENT_SECRET, SP_TENANT_ID from
#   env vars injected at serving time via ace-secrets scope). If any SP
#   credential is missing (e.g. in notebook/dev context), it falls back
#   to DATABRICKS_TOKEN (the injected M2M token or notebook session token).
#   This dual-path approach lets the same class work in both serving and
#   notebook verification (Cell 7) without code changes.
#
# CITATIONS AS JSON STRING:
#   citations is serialised as json.dumps(list) in predict() output.
#   MLflow schema enforcement rejects list-of-dicts output columns but
#   accepts STRING. Cell 7 and the Supervisor both json.loads() it.
#
# MLflow SIGNATURE:
#   Built via infer_signature() on typed sample dicts where citations
#   is already a JSON string. Avoids ModelSignature constructor issues
#   on older MLflow versions and ensures the schema is enforced at
#   serving time.
#
# resources= BLOCK:
#   Declares DatabricksVectorSearchIndex and DatabricksServingEndpoint
#   resources. This grants the M2M token auto-injected by Databricks
#   serving runtime the permission to call these resources. Without
#   this block, the serving container cannot reach VS or the LLM.
#
# CHAMPION ALIAS:
#   After registration, the champion alias is set on the new version.
#   Cell 8 (verification) and Cell 9 (deployment) reference the model
#   by alias, not by version number, so they always use the latest
#   promoted version without needing to be updated manually.
# ================================================================

import json
import time
import mlflow
from mlflow.models import infer_signature
from mlflow.models.resources import (
    DatabricksVectorSearchIndex,
    DatabricksServingEndpoint,
)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)


class RagAgentModel(mlflow.pyfunc.PythonModel):

    # -- Constants ------------------------------------------------
    VS_ENDPOINT_NAME  = "ace-chat-vs-endpoint"
    VS_INDEX_NAME     = "ace_theorem.chat.policy_documents_index"
    LLM_ENDPOINT      = "databricks-claude-sonnet-4-5"
    NUM_RESULTS       = 4      # Chunks returned per similarity search (reduced from 7 for latency)
    MAX_TOKENS        = 1000    # Max LLM output tokens per response (reduced from 1024 for latency)
    DATABRICKS_SCOPE  = "2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default"

    SYSTEM_PROMPT = """You are a lending policy expert for Vantage Bank. Your role is to answer questions using only the provided policy excerpts and relevant prior conversation context.

## Rules
1. Use only the provided excerpts — never use general banking knowledge or outside information.
2. If the question is unrelated to bank lending policy, decline to answer.
3. Cite every factual statement with [Source N].
4. Copy figures (LTV, DTI, thresholds, dates, limits, approval levels) exactly — no rounding, rewriting, or paraphrasing.
5. If the answer is not covered by the excerpts, state exactly: "The available policy documents do not cover this. Please refer to the appropriate policy owner or the Compliance team."
6. If sources contradict, state exactly: "Note: [Source N] and [Source M] contain conflicting guidance on this point."
7. Use the document's exact wording, including policy-strength terms such as must, should, may, and requires.
8. Never speculate, infer unstated requirements, or soften policy language.
9. Do not cite a source unless it directly supports the statement it follows.
10. If the answer depends on missing context not provided in the excerpts, state that the available excerpts are insufficient to determine the answer definitively.
11. Use prior conversation only to clarify references — never as policy evidence and never in place of the provided excerpts.
12. If multiple consecutive statements come from the same document and section, cite [Source N] once at the end of the last statement in that group — do not repeat the same citation after every line.
13. When your answer involves a policy threshold, eligibility criterion, or requirement, explicitly list the loan data fields needed to evaluate compliance -- for example: "To evaluate this policy: loan amount, collateral type, Phase I Assessment status." This allows the synthesis layer to connect policy requirements to available loan data.

## Response format

**Answer**
- Give a direct answer in 1–2 sentences.
- If the answer is partial, conditional, or limited by missing information, say that clearly.

**Policy Basis**
- Present supporting policy points as bullets.
- Use one bullet per requirement, exception, threshold, approval rule, prohibition, or condition. [Source N]

**Key Figures** *(omit if none)*
- Metric: exact value [Source N]
- Copy numeric values exactly as written.

**Limitations / Missing Context** *(omit if none)*

**Conflicting Guidance** *(omit if none)*
- Note: [Source N] and [Source M] contain conflicting guidance on this point.

**Sources**
- [Source N]: Document name — Section name
"""

    # -- load_context: runs once when the container starts --------
    def load_context(self, context):
        import os as _os, mlflow.deployments
        from msal import ConfidentialClientApplication
        self._host  = _os.environ.get("DATABRICKS_HOST", "").rstrip("/")
        self.client = mlflow.deployments.get_deploy_client("databricks")
        _tenant = _os.environ.get("SP_TENANT_ID", "")
        _cid    = _os.environ.get("SP_CLIENT_ID", "")
        _csec   = _os.environ.get("SP_CLIENT_SECRET", "")
        _has_sp = bool(_tenant and _cid and _csec)
        if _has_sp:
            self._msal_app = ConfidentialClientApplication(
                client_id=_cid,
                client_credential=_csec,
                authority=f"https://login.microsoftonline.com/{_tenant}"
            )
        else:
            self._msal_app = None
        print(f"[ACE] host={bool(self._host)} sp_vars={_has_sp}")

    # -- Get a fresh Azure AD token on every call (1 hr expiry) --
    def _get_token(self):
        """Return a valid SP OAuth token via MSAL (cached + auto-refreshed).
        Falls back to DATABRICKS_TOKEN for notebook/dev context.
        """
        import os as _os
        if self._msal_app is not None:
            result = self._msal_app.acquire_token_for_client(
                scopes=[self.DATABRICKS_SCOPE]
            )
            if "access_token" in result:
                return result["access_token"]
        return (
            _os.environ.get("DATABRICKS_TOKEN", "") or
            _os.environ.get("DB_TOKEN", "")
        )

    # -- predict: called on every request -------------------------
    def predict(self, context, model_input):
        import json
        from databricks.vector_search.client import VectorSearchClient

        question = (
            model_input["question"]
            if isinstance(model_input, dict)
            else model_input["question"].iloc[0]
        )

        # Step 1: Get fresh SP token and query Vector Search
        token = self._get_token()
        vsc   = VectorSearchClient(
            workspace_url         = self._host,
            personal_access_token = token,
            disable_notice        = True
        )
        index   = vsc.get_index(self.VS_ENDPOINT_NAME, self.VS_INDEX_NAME)
        results = index.similarity_search(
            query_text  = question,
            columns     = ["doc_name", "section", "chunk_text", "page_num"],
            num_results = self.NUM_RESULTS
        )
        hits = results.get("result", {}).get("data_array", [])

        if not hits:
            return {
                "answer"   : "I could not find relevant policy information.",
                "citations": json.dumps([]),
                "question" : question
            }

        # Step 2: Build context with attribution
        parts, citations = [], []
        for i, hit in enumerate(hits, 1):
            doc_name, section, chunk_text, page_num = hit[0], hit[1], hit[2], (hit[3] if len(hit) > 3 else 0)
            parts.append(
                f"[Source {i}]\n"
                f"Document : {doc_name}\n"
                f"Section  : {section}\n"
                f"Content  : {chunk_text}"
            )
            citations.append({"source_num": i, "doc_name": doc_name, "section": section, "page_num": page_num or 0, "chunk_text": chunk_text})

        context_text = "\n---\n".join(parts)

        # Step 3: Call Claude Haiku 4.5
        response = self.client.predict(
            endpoint = self.LLM_ENDPOINT,
            inputs   = {
                "messages": [
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user",   "content": (
                        f"{context_text}\n\n"
                        f"Question: {question}\n\n"
                        "Cite [Source N] for every fact."
                    )}
                ],
                "max_tokens" : self.MAX_TOKENS,
                "temperature": 0.0
            }
        )
        answer = response["choices"][0]["message"]["content"]

        # citations serialised as JSON string -- MLflow schema safe
        return {
            "answer"   : answer,
            "citations": json.dumps(citations),
            "question" : question
        }


# -- Signature via infer_signature on typed sample dicts ----------
sample_input = {
    "question": "What are the LTV limits for commercial real estate?"
}
sample_output = {
    "answer"   : "Sample answer text.",
    "citations": json.dumps([{"source_num": 1, "doc_name": "sample_doc", "section": "sample_section"}]),
    "question" : "What are the LTV limits for commercial real estate?"
}
signature = infer_signature(sample_input, sample_output)

# -- Log and register ----------------------------------------------
with mlflow.start_run(run_name="rag_agent_registration"):

    try:
        model_info = mlflow.pyfunc.log_model(
            name                  = "rag_agent",
            python_model          = RagAgentModel(),
            registered_model_name = REGISTERED_MODEL,
            signature             = signature,
            pip_requirements      = [
                "mlflow",
                "databricks-vectorsearch",
                "cloudpickle",
                "msal",
            ],
            resources = [
                DatabricksVectorSearchIndex(index_name="ace_theorem.chat.policy_documents_index"),
                DatabricksServingEndpoint(endpoint_name="databricks-claude-haiku-4-5"),
            ],
        )
        model_uri = model_info.model_uri
    except AttributeError:
        # MLflow post-log validation bug -- model was registered successfully
        run = mlflow.active_run()
        model_uri = f"runs:/{run.info.run_id}/rag_agent"
    print(f"Model logged: {model_uri}")

# -- Set champion alias --------------------------------------------
client_uc = mlflow.tracking.MlflowClient()
print("Waiting for model version to be ready ...")
time.sleep(10)

versions = client_uc.search_model_versions(f"name='{REGISTERED_MODEL}'")
if not versions:
    raise RuntimeError("No model versions found -- registration may have failed.")

version = versions[0].version
client_uc.set_registered_model_alias(
    name    = REGISTERED_MODEL,
    alias   = MODEL_ALIAS,
    version = version
)

print(f"Registered : {REGISTERED_MODEL}")
print(f"Version    : {version}")
print(f"Alias      : {MODEL_ALIAS}")
print("Registration complete. Proceed to Cell 8 to verify, then Cell 9 to deploy.")


In [ ]:
# ================================================================
# CELL 4 -- Deploy ace-rag-agent-endpoint
# ================================================================
#
# PURPOSE:
#   Deploys the registered RagAgentModel as a live Databricks Model
#   Serving endpoint. After this cell completes, the Supervisor can
#   call this endpoint via HTTP to get policy Q&A answers.
#
# WHY A SEPARATE HTTP ENDPOINT:
#   Loading a model via mlflow.pyfunc.load_model() inside a serving
#   container causes PERMISSION_DENIED because the container lacks
#   credentials to pull from Unity Catalog. Deploying as an independent
#   endpoint avoids this -- the Supervisor makes an HTTP call instead.
#
# ENVIRONMENT VARIABLES INJECTED:
#   DATABRICKS_HOST    -- workspace URL. Required by VectorSearchClient
#                         when authenticating with SP credentials.
#                         Not reliably auto-set in all serving runtimes.
#   SP_CLIENT_ID       -- from ace-secrets scope (Databricks Secrets)
#   SP_CLIENT_SECRET   -- from ace-secrets scope
#   SP_TENANT_ID       -- from ace-secrets scope
#   These are read by _get_token() to get an Azure AD token for
#   Vector Search via MSAL client_credentials flow.
#
# CREATE OR UPDATE BEHAVIOUR (idempotent):
#   404 on GET  -> creates endpoint (POST)
#   200 on GET  -> updates config   (PUT)
#   Safe to re-run when deploying a new model version.
#
# SCALING CONFIG:
#   workload_size = Small
#   scale_to_zero = False (keeps endpoint warm; eliminates 30-60s cold-start latency)
#
# POLLING:
#   Checks every 30s, times out after 30 min (60 attempts).
#   New endpoint: typically READY in 5-10 min.
#   Config update: typically READY in 3-5 min.
# ================================================================

import requests, time, mlflow

host    = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
token   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

ENDPOINT_NAME = "ace-rag-agent-endpoint"

# Get the latest registered version
mlflow.set_registry_uri("databricks-uc")
client_uc = mlflow.tracking.MlflowClient()
versions  = client_uc.search_model_versions(f"name='{REGISTERED_MODEL}'")
if not versions:
    raise RuntimeError("No model versions found. Run Cell 7 first.")
latest_version = versions[0].version
print(f"Deploying {REGISTERED_MODEL} v{latest_version} ' {ENDPOINT_NAME}")

# Endpoint payload
# DATABRICKS_HOST is computed at notebook run time and passed as a
# literal string. VectorSearchClient inside the container requires it
# when authenticating with SP credentials via MSAL.
endpoint_config = {
    "served_models": [{
        "name"                 : "rag-agent",
        "model_name"           : REGISTERED_MODEL,
        "model_version"        : latest_version,
        "workload_size"        : "Small",
        "scale_to_zero_enabled": False,
        "environment_vars"     : {
            "DATABRICKS_HOST" : host,
            "SP_CLIENT_ID"    : "{{secrets/ace-secrets/sp-client-id}}",
            "SP_CLIENT_SECRET": "{{secrets/ace-secrets/sp-client-secret}}",
            "SP_TENANT_ID"    : "{{secrets/ace-secrets/sp-tenant-id}}",
        }
    }]
}

# Create or update
check = requests.get(
    f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
    headers=headers
)

if check.status_code == 404:
    print("Endpoint does not exist -- creating ...")
    r = requests.post(
        f"{host}/api/2.0/serving-endpoints",
        headers=headers,
        json={"name": ENDPOINT_NAME, "config": endpoint_config}
    )
    r.raise_for_status()
    print("Create request accepted.")
else:
    print("Endpoint exists -- updating config ...")
    r = requests.put(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
        headers=headers,
        json=endpoint_config
    )
    r.raise_for_status()
    print("Update request accepted.")

# Poll until READY
print("\nPolling for READY state (max 30 min) ...")
for attempt in range(60):
    state_resp = requests.get(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
        headers=headers
    )
    state_resp.raise_for_status()
    state         = state_resp.json().get("state", {})
    ready_state   = state.get("ready", "")
    config_update = state.get("config_update", "NOT_UPDATING")
    print(f"  [{attempt+1:02d}] ready={ready_state}  config_update={config_update}")
    if ready_state == "READY" and config_update in ("NOT_UPDATING", ""):
        break
    time.sleep(30)
else:
    raise RuntimeError(f"{ENDPOINT_NAME} did not become READY within 30 minutes.")

print("=" * 55)
print(f"Endpoint   : {ENDPOINT_NAME}")
print(f"Model      : {REGISTERED_MODEL} v{latest_version}")
print(f"Status     : READY")
print("=" * 55)
print("ace-rag-agent-endpoint is READY.")
print("Proceed to 04_page_context_agent.ipynb")